# Whisper Pularr on Google Colab T4

This notebook follows the promotion-aware Colab flow: decode sweep and evaluation first, supervised continuation second, then optional self-training.

## 1. Mount Google Drive

Drive is used automatically when mounted, so checkpoints and reports survive Colab restarts.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 2. Clone the repository

In [ ]:
%cd /content
!rm -rf /content/whisper-pularr
!git clone https://github.com/zainmouhamed456-hub/Wispher_Pularr.git whisper-pularr
%cd /content/whisper-pularr


## 3. Install dependencies

In [ ]:
%cd /content
!test -d /content/whisper-pularr || git clone https://github.com/zainmouhamed456-hub/Wispher_Pularr.git /content/whisper-pularr
%cd /content/whisper-pularr
!bash /content/whisper-pularr/colab/bootstrap_t4_free.sh


## 4. Check the GPU

In [ ]:
!nvidia-smi


## 5. Configure the session

Use the same config cell for every session. For Session 1 keep `COLAB_EVAL_ONLY = "1"`. For Session 2 switch it to `"0"` and leave `COLAB_RUN_SELF_TRAIN = "0"`. For Session 3 keep `COLAB_EVAL_ONLY = "0"` and raise `COLAB_MAX_TRAIN_SAMPLES` to `"4096"`. For optional Stage 2 set `COLAB_RUN_SELF_TRAIN = "1"`.

In [ ]:
import json
import os
import signal
import subprocess
import sys
import time
from pathlib import Path
from IPython.display import Markdown, display

REPO_ROOT = "/content/whisper-pularr"
DATASET_NAME = "google/WaxalNLP"
DATASET_CONFIG = "ful_asr"
MODEL_ID = "openai/whisper-small"
TEACHER_MODEL = "openai/whisper-large-v3"
WHISPER_LANGUAGE = ""
AUX_DATASET_NAME = ""
AUX_DATASET_CONFIG = ""

COLAB_EVAL_ONLY = "1"
COLAB_BASE_MODEL = ""
COLAB_RESUME_FROM = ""
COLAB_COMPARE_BEAMS = "1,3"
COLAB_FIXED_SLICE_SIZE = "64"
COLAB_MAX_TRAIN_SAMPLES = "1024"
COLAB_MAX_EVAL_SAMPLES = "64"
COLAB_FULL_VALIDATION_MAX_SAMPLES = "256"
COLAB_SUPERVISED_EPOCHS = "1"
COLAB_EARLY_STOP_PATIENCE = "1"
COLAB_RUN_SELF_TRAIN = "0"
COLAB_PSEUDO_BEAM_SIZE = "3"
COLAB_PSEUDO_BATCH_SIZE = "2"
COLAB_MAX_KEEP_MULTIPLE = "0.75"
COLAB_MANIFEST_EVERY = "4000"
RUN_PIPELINE_HELPER_VERSION = "2026-03-27-stream-v2"

if Path("/content/drive/MyDrive").exists():
    RUNS_ROOT = "/content/drive/MyDrive/whisper-pularr-runs"
else:
    RUNS_ROOT = "/content/whisper-pularr-runs"

def run_pipeline():
    env = os.environ.copy()
    env.update(
        {
            "RUNS_ROOT": RUNS_ROOT,
            "HF_HOME": os.environ.get("HF_HOME", "/content/hf-cache"),
            "PYTHONUNBUFFERED": "1",
            "USE_TF": "0",
            "TRANSFORMERS_NO_TF": "1",
            "TOKENIZERS_PARALLELISM": "false",
            "COLAB_EVAL_ONLY": COLAB_EVAL_ONLY,
            "COLAB_BASE_MODEL": COLAB_BASE_MODEL,
            "COLAB_RESUME_FROM": COLAB_RESUME_FROM,
            "COLAB_COMPARE_BEAMS": COLAB_COMPARE_BEAMS,
            "COLAB_FIXED_SLICE_SIZE": COLAB_FIXED_SLICE_SIZE,
            "COLAB_MAX_TRAIN_SAMPLES": COLAB_MAX_TRAIN_SAMPLES,
            "COLAB_MAX_EVAL_SAMPLES": COLAB_MAX_EVAL_SAMPLES,
            "COLAB_SUPERVISED_EPOCHS": COLAB_SUPERVISED_EPOCHS,
            "COLAB_EARLY_STOP_PATIENCE": COLAB_EARLY_STOP_PATIENCE,
            "COLAB_FULL_VALIDATION_MAX_SAMPLES": COLAB_FULL_VALIDATION_MAX_SAMPLES,
            "COLAB_RUN_SELF_TRAIN": COLAB_RUN_SELF_TRAIN,
            "COLAB_PSEUDO_BEAM_SIZE": COLAB_PSEUDO_BEAM_SIZE,
            "COLAB_PSEUDO_BATCH_SIZE": COLAB_PSEUDO_BATCH_SIZE,
            "COLAB_MAX_KEEP_MULTIPLE": COLAB_MAX_KEEP_MULTIPLE,
            "COLAB_MANIFEST_EVERY": COLAB_MANIFEST_EVERY,
        }
    )
    command = [
        sys.executable,
        f"{REPO_ROOT}/colab/run_t4_free.py",
        REPO_ROOT,
        DATASET_NAME,
        DATASET_CONFIG,
        MODEL_ID,
        TEACHER_MODEL,
        WHISPER_LANGUAGE,
        AUX_DATASET_NAME,
        AUX_DATASET_CONFIG,
    ]
    mode = "eval-only" if COLAB_EVAL_ONLY == "1" else ("self-train" if COLAB_RUN_SELF_TRAIN == "1" else "supervised")
    display(Markdown(f"**run_pipeline helper version:** `{RUN_PIPELINE_HELPER_VERSION}`"))
    display(Markdown(f"**Starting {mode} pipeline.** Session 1 eval-only runs can take several minutes on a free Colab T4."))
    print(f"[run_pipeline output] helper version: {RUN_PIPELINE_HELPER_VERSION}", flush=True)
    print(f"[run_pipeline output] Starting {mode} pipeline. Session 1 eval-only runs can take several minutes on a free Colab T4.", flush=True)
    process = subprocess.Popen(
        command,
        env=env,
        start_new_session=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    try:
        while True:
            if process.stdout is None:
                break
            line = process.stdout.readline()
            if line:
                print(line, end="", flush=True)
                continue
            return_code = process.poll()
            if return_code is not None:
                break
            time.sleep(0.2)
    except KeyboardInterrupt:
        print("[run_pipeline output] Interrupted from the notebook. Terminating the child pipeline process...", flush=True)
        try:
            os.killpg(os.getpgid(process.pid), signal.SIGTERM)
        except ProcessLookupError:
            pass
        except OSError:
            process.terminate()
        try:
            process.wait(timeout=10)
        except subprocess.TimeoutExpired:
            try:
                os.killpg(os.getpgid(process.pid), signal.SIGKILL)
            except ProcessLookupError:
                pass
            except OSError:
                process.kill()
            process.wait()
        print("[run_pipeline output] Pipeline interrupted cleanly. You can rerun the same cell when ready.", flush=True)
        return
    finally:
        if process.stdout is not None:
            process.stdout.close()
    return_code = process.poll()
    if return_code is None:
        time.sleep(0.2)
        return_code = process.poll()
    if return_code != 0:
        raise subprocess.CalledProcessError(return_code, command)

def show_json(path: Path):
    if path.exists():
        print(f"\n== {path} ==")
        print(path.read_text(encoding="utf-8"))
    else:
        print(f"Missing: {path}")

print(json.dumps({
    "RUNS_ROOT": RUNS_ROOT,
    "COLAB_EVAL_ONLY": COLAB_EVAL_ONLY,
    "COLAB_BASE_MODEL": COLAB_BASE_MODEL,
    "COLAB_RESUME_FROM": COLAB_RESUME_FROM,
    "COLAB_COMPARE_BEAMS": COLAB_COMPARE_BEAMS,
    "COLAB_FIXED_SLICE_SIZE": COLAB_FIXED_SLICE_SIZE,
    "COLAB_MAX_TRAIN_SAMPLES": COLAB_MAX_TRAIN_SAMPLES,
    "COLAB_MAX_EVAL_SAMPLES": COLAB_MAX_EVAL_SAMPLES,
    "COLAB_FULL_VALIDATION_MAX_SAMPLES": COLAB_FULL_VALIDATION_MAX_SAMPLES,
    "COLAB_RUN_SELF_TRAIN": COLAB_RUN_SELF_TRAIN,
    "RUN_PIPELINE_HELPER_VERSION": RUN_PIPELINE_HELPER_VERSION,
}, indent=2))


## 6. Session 1: beam sweep and evaluation only

In [ ]:
COLAB_EVAL_ONLY = "1"
COLAB_RUN_SELF_TRAIN = "0"
print(f"[section 6] using helper {RUN_PIPELINE_HELPER_VERSION}", flush=True)
print("[section 6] launching eval-only pipeline...", flush=True)
run_pipeline()


## 7. Session 2 or 3: supervised continuation

Leave `COLAB_BASE_MODEL` empty to reuse the promoted checkpoint automatically. Set `COLAB_RESUME_FROM` only when resuming a saved trainer checkpoint such as `.../checkpoint-125`.

In [ ]:
COLAB_EVAL_ONLY = "0"
COLAB_RUN_SELF_TRAIN = "0"
print(f"[section 7] using helper {RUN_PIPELINE_HELPER_VERSION}", flush=True)
print("[section 7] launching supervised pipeline...", flush=True)
run_pipeline()


## 8. Inspect decode selection and promotions

In [ ]:
runs_root = Path(RUNS_ROOT)
show_json(runs_root / "reports" / "colab_decode_selection.json")
show_json(runs_root / "reports" / "colab_promotion_summary.json")
for path in sorted((runs_root / "reports").glob("*.json")):
    print(path)


## 9. Optional Stage 2 self-training

Run this only after the supervised checkpoint is stable.

In [ ]:
COLAB_EVAL_ONLY = "0"
COLAB_RUN_SELF_TRAIN = "1"
print(f"[section 9] using helper {RUN_PIPELINE_HELPER_VERSION}", flush=True)
print("[section 9] launching self-train pipeline...", flush=True)
run_pipeline()
